In [4]:
# --- Fill here on Colab (or use Secrets: LLM_DOC_REPO_URL, HF_TOKEN) ---
REPO_GIT_URL = "https://github.com/AramAleksanyan/llm_doc_classification.git"  # HTTPS git URL of this project, e.g. "https://github.com/you/llm_doc_classification.git"
HF_TOKEN = ""  # Use Colab Secret / env HF_TOKEN only — never paste tokens into the notebook

import os
import subprocess
import sys
from pathlib import Path

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _has_package(root: Path) -> bool:
    return (root / "src" / "llm_doc_classification").is_dir()


if not _in_colab():
    print("Not on Colab — clone skipped. Paste HF_TOKEN above for Gemma, or set env HF_TOKEN.")
else:
    REPO_DIR = Path("/content/llm_doc_classification").resolve()
    repo_url = (REPO_GIT_URL or os.environ.get("LLM_DOC_REPO_URL", "")).strip()
    if not _has_package(REPO_DIR):
        if not repo_url:
            raise RuntimeError(
                "Set REPO_GIT_URL in this cell or Colab secret LLM_DOC_REPO_URL, then re-run. "
                "If you use Drive/upload only: %cd into the project folder, "
                "run os.environ['LLM_DOC_REPO'] = '<that path>', then skip re-running this cell."
            )
        if REPO_DIR.is_dir():
            subprocess.run(["rm", "-rf", str(REPO_DIR)], check=False)
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(REPO_DIR)], check=True)
    if not _has_package(REPO_DIR):
        raise RuntimeError(f"Missing src/llm_doc_classification under {REPO_DIR}")
    os.chdir(REPO_DIR)
    os.environ["LLM_DOC_REPO"] = str(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
        check=True,
    )
    print("LLM_DOC_REPO =", os.environ["LLM_DOC_REPO"])


RuntimeError: Missing src/llm_doc_classification under /content/llm_doc_classification

### Google Colab — run order

1. **Runtime → Change runtime type → GPU** (Gemma 2 2B needs a GPU; T4 or better is safest).
2. **Secrets (optional but recommended):** add `HF_TOKEN` with a [Hugging Face access token](https://huggingface.co/settings/tokens) after accepting the Gemma license on the model page. You can put the git HTTPS URL in secret **`LLM_DOC_REPO_URL`** instead of pasting it in the next cell.
3. **Next code cell:** set **`REPO_GIT_URL`** to your repo’s **HTTPS** clone URL (full tree: `src/`, `data/`, `configs/`, …), then run it once. It clones to `/content/llm_doc_classification`, sets `LLM_DOC_REPO`, and installs `requirements.txt`.
4. Run the remaining cells **top to bottom**.

If the repo is only on your machine, push it to GitHub/GitLab first, or upload a zip to Drive and open the project from there, then set **`LLM_DOC_REPO`** to that folder path.


In [1]:
# Colab bootstrap: clone repo, cd, pip deps. Local Jupyter: not on Colab → no-op.
#
# Set ONE of:
#   • REPO_GIT_URL below (HTTPS), or
#   • Colab secret / env LLM_DOC_REPO_URL with the same URL.
REPO_GIT_URL = ""  # e.g. "https://github.com/youruser/llm_doc_classification.git"

import os
import subprocess
import sys
from pathlib import Path


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _has_package(root: Path) -> bool:
    return (root / "src" / "llm_doc_classification").is_dir()


if not _in_colab():
    print("Not on Colab — skipped. Local: run from repo root or set LLM_DOC_REPO.")
else:
    REPO_DIR = Path("/content/llm_doc_classification").resolve()
    repo_url = (REPO_GIT_URL or os.environ.get("LLM_DOC_REPO_URL", "")).strip()
    if not _has_package(REPO_DIR):
        if not repo_url:
            raise RuntimeError(
                "Paste your HTTPS git URL into REPO_GIT_URL in this cell, "
                "or set Colab secret LLM_DOC_REPO_URL, then re-run."
            )
        if REPO_DIR.is_dir():
            subprocess.run(["rm", "-rf", str(REPO_DIR)], check=False)
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(REPO_DIR)], check=True)
    if not _has_package(REPO_DIR):
        raise RuntimeError(f"Repo missing src/llm_doc_classification under {REPO_DIR}")
    os.chdir(REPO_DIR)
    os.environ["LLM_DOC_REPO"] = str(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
        check=True,
    )
    print("LLM_DOC_REPO =", os.environ["LLM_DOC_REPO"])


RuntimeError: Paste your HTTPS git URL into REPO_GIT_URL in this cell, or set Colab secret LLM_DOC_REPO_URL, then re-run.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer


def _has_package(p: Path) -> bool:
    return (p / "src" / "llm_doc_classification").is_dir()


def _find_repo_root() -> Path:
    for key in ("LLM_DOC_REPO", "COLAB_REPO_ROOT"):
        raw = os.environ.get(key)
        if not raw:
            continue
        p = Path(raw).expanduser().resolve()
        if not p.is_dir():
            continue  # stale env on Colab / wrong path — fall back to auto-detect
        if _has_package(p):
            return p
        raise RuntimeError(
            f"{key}={p} exists but is missing src/llm_doc_classification/. "
            "Upload/clone the full repository (including the src/ tree), or fix the path."
        )

    cur = Path.cwd().resolve()
    for p in (cur, *cur.parents):
        if _has_package(p):
            return p

    for base in (Path("/content"), Path("/content/drive/MyDrive")):
        if not base.is_dir():
            continue
        try:
            for sub in base.iterdir():
                if sub.is_dir() and _has_package(sub):
                    return sub.resolve()
        except OSError:
            continue

    try:
        for sub in cur.iterdir():
            if sub.is_dir() and _has_package(sub):
                return sub.resolve()
    except OSError:
        pass

    raise RuntimeError(
        "Cannot find project root (need src/llm_doc_classification/). "
        "On Colab: %cd into the cloned repo, or set LLM_DOC_REPO to that folder."
    )


REPO = _find_repo_root()
sys.path.insert(0, str(REPO / "src"))

MODEL_ID = "google/gemma-2-2b-it"
TASK_JSON = REPO / "configs" / "prompting" / "7domain_labels.json"
TEST_CSV = REPO / "data" / "multidomain_documents" / "test.csv"
TRAIN_CSV = REPO / "data" / "multidomain_documents" / "train.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from llm_doc_classification.prompting import FewShotSpec, PromptBuilder, PromptTaskSpec

raw = json.loads(TASK_JSON.read_text(encoding="utf-8"))
LABELS = tuple(raw["labels"])
text_col, label_col = raw["text_column"], raw["label_column"]

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

task = PromptTaskSpec(text_column=text_col, label_column=label_col, label_names=LABELS)
builder = PromptBuilder(task, FewShotSpec(n_examples=0, seed=42))

In [ ]:
_hf_token = os.environ.get("HF_TOKEN")  # first cell HF_TOKEN, Secret, or env

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=_hf_token)
model_dtype = torch.bfloat16 if device.type == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=_hf_token,
    dtype=model_dtype,
    low_cpu_mem_usage=True,
)
model.to(device)
model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def parse_to_label(generated: str, labels: tuple[str, ...]) -> str:
    s = generated.strip().splitlines()[0].strip()
    low = generated.lower()
    for lab in labels:
        if s == lab:
            return lab
    slow = s.lower()
    for lab in labels:
        if slow == lab.lower():
            return lab
    for lab in labels:
        if lab in s:
            return lab
    for lab in labels:
        if lab.lower() in slow:
            return lab
    for lab in sorted(labels, key=len, reverse=True):
        if lab.lower() in low:
            return lab
    return labels[0]


MAX_PROMPT_TOKENS = 1024  # lower if you still hit CUDA OOM (long docs × attention)

y_true: list[str] = []
y_pred: list[str] = []

for _, row in test_df.iterrows():
    if pd.isna(row.get(text_col)) or pd.isna(row.get(label_col)):
        continue
    true_lab = str(row[label_col])
    if true_lab not in LABELS:
        continue
    doc = str(row[text_col])
    built = builder.build(train_df, doc, template_name="zero_shot.txt")
    messages = [{"role": "user", "content": built.text}]
    batch = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    input_ids = batch["input_ids"].to(device)
    attn = batch.get("attention_mask")
    if attn is not None:
        attn = attn.to(device)
    with torch.inference_mode():
        out = model.generate(
            input_ids,
            attention_mask=attn,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen = tokenizer.decode(out[0, input_ids.shape[1] :], skip_special_tokens=True)
    y_true.append(true_lab)
    y_pred.append(parse_to_label(gen, LABELS))

In [ ]:
lab_list = list(LABELS)
print("accuracy", float(accuracy_score(y_true, y_pred)))
print(
    "macro_f1",
    float(f1_score(y_true, y_pred, average="macro", labels=lab_list, zero_division=0)),
)